# Laboratório: MCP

**Grupo:**

- Nome 1:
- Nome 2:

Neste laboratório deverão ser construídos um servidor MCP e uma skill para um domínio escolhido pelo grupo. O servidor é publicado no mural da turma e usado por um agente que também se conecta aos servidores MCP dos outros grupos, e a skill ensina esse agente a fazer uma tarefa do domínio com as ferramentas do servidor. O domínio pode ser o mesmo do laboratório de RAG ou do agente feito antes, e nesse caso o retriever ou as ferramentas construídas lá viram ferramentas do servidor, ou pode ser um domínio novo. Cada seção descreve o que se espera dela e traz um espaço para o código, e podem ser criadas quantas células forem precisas.

O servidor vai ser usado por agentes que o grupo não controla, com modelos e mensagens de sistema diferentes. Tudo o que esses agentes sabem sobre o servidor vem dos nomes, das docstrings e dos parâmetros, então essas descrições devem ser escritas para quem nunca viu o código.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph "langchain[mcp]"
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import os
import re
import subprocess
import sys
import time
from pathlib import Path

os.environ["FASTMCP_LOG_LEVEL"] = "WARNING"

import pandas as pd
import yaml

from fastmcp import Client
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage
from langchain.tools import tool

## Servidor

O servidor deve ser escrito com o `FastMCP` em um arquivo `server.py`, criado pelo grupo na mesma pasta do notebook e entregue junto com ele. Ele deve oferecer:

- **Ferramentas:** o retriever do RAG ou as ferramentas do agente anterior, se o domínio for o mesmo, e pelo menos duas outras ferramentas, que podem consultar uma API pública, calcular algo ou ler dados próprios do domínio.
- **Recurso:** pelo menos um, com URI fixa ou template, com dados que a aplicação anexa ao contexto.
- **Prompt:** pelo menos um, com argumentos.

O arquivo segue este formato:

```python
from fastmcp import FastMCP

mcp = FastMCP("nome-do-servidor")

# Ferramentas, recursos e prompts aqui.

if __name__ == "__main__":
    mcp.run(transport="http", host="0.0.0.0", port=8000, show_banner=False)
```

O servidor roda como um processo separado do notebook, então não enxerga as variáveis definidas aqui: tudo de que ele precisa, como o modelo de embedding e o caminho do índice, deve ser criado dentro do arquivo. No `mcp.run`, o `host="0.0.0.0"` permite que o servidor aceite as requisições que chegam pela URL pública, e a porta deve ser a mesma usada na seção de publicação.

### Índice do RAG

Se o servidor usa o retriever, o índice do vector store precisa existir na pasta em que o servidor roda. No Colab, a pasta do índice pode ser enviada compactada e descompactada aqui, ou o índice pode ser refeito com o código de indexação do laboratório de RAG. O modelo de embedding do servidor deve ser o mesmo usado para criar o índice.

## Publicação

A célula abaixo inicia o servidor e um túnel do Cloudflare, que cria uma URL pública em `trycloudflare.com` apontando para a porta local, sem exigir conta. A célula espera o endereço novo se propagar no DNS, o que leva de alguns segundos a um minuto, e os dois processos continuam rodando enquanto o ambiente de execução estiver ativo. Depois de alterar o `server.py`, basta rodar a célula de novo: ela encerra os processos anteriores e gera uma URL nova.

In [ ]:
PORT = 8000

for process in globals().get("processes", []):
    process.kill()

processes = [
    subprocess.Popen([sys.executable, "server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT),
    subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT),
]

while not re.search(r"Registered tunnel connection", open("tunnel.log").read()):
    time.sleep(1)

PUBLIC_URL = re.search(r"https://[\w-]+\.trycloudflare\.com", open("tunnel.log").read()).group(0) + "/mcp"

time.sleep(10)
while subprocess.run(["curl", "-s", "-o", "/dev/null", PUBLIC_URL]).returncode != 0:
    time.sleep(5)

print(PUBLIC_URL)

Se algo falhar, os logs dos dois processos ficam em `server.log` e `tunnel.log`. Um erro de sintaxe no `server.py` aparece no primeiro, e o servidor para de responder sem nenhum aviso no notebook.

In [ ]:
print(open("server.log").read()[-3000:])

A URL é pública e não tem autenticação: qualquer pessoa que a conheça pode chamar as ferramentas, e o custo das chamadas ao modelo de embedding sai da chave do grupo. Por isso o servidor não deve oferecer ferramentas que gravem arquivos, executem código ou exponham dados pessoais.

### Teste pela URL pública

Antes de ser publicado no mural, o servidor deve ser testado pela URL pública, com um cliente que liste as ferramentas, os recursos e os prompts e que chame cada ferramenta pelo menos uma vez, incluindo um caso com argumento inválido. O `MCPAdapter` converte só as ferramentas; os recursos e os prompts são listados e lidos com o `Client` do FastMCP, que recebe a mesma URL. A URL só deve ser publicada depois que todas as chamadas funcionarem.

In [ ]:
async with MCPAdapter(PUBLIC_URL) as adapter:
    own_tools = await adapter.list_tools()

pd.DataFrame([{"name": tool.name, "description": tool.description, "args": list(tool.args)} for tool in own_tools])

## Cliente

O cliente é um agente conectado ao servidor do grupo e a pelo menos dois servidores MCP do mural, cada um adicionado ao dicionário `mcpServers` com a linha pronta do cartão do servidor. O servidor do grupo é acessado pelo endereço local, que continua o mesmo quando a URL pública muda. As URLs dos outros grupos deixam de funcionar quando eles encerram o túnel, então as saídas das células que as usam devem ficar salvas no notebook. Com mais de um servidor, o `MCPAdapter` prefixa o nome de cada ferramenta com a chave do servidor, o que evita colisão quando dois servidores têm ferramentas com o mesmo nome. Cada ferramenta abre a conexão quando é chamada, e chamar o agente dentro de `async with adapter` faz as chamadas de uma pergunta compartilharem uma conexão.

In [ ]:
servers = {
    "nosso": {"url": f"http://localhost:{PORT}/mcp"},
    # "receitas": {"url": "https://....trycloudflare.com/mcp"},
}

adapter = MCPAdapter({"mcpServers": servers})

### Skill

A skill é uma pasta em `skills/` com um `SKILL.md`, e a pasta e o arquivo devem ser criados pelo grupo: o frontmatter traz `name` e `description`, e o corpo traz as instruções. Ela deve ensinar uma tarefa do domínio que precise de vários passos, e não repetir o que uma ferramenta já faz sozinha:

- **Descrição:** diz o que a skill faz e quando usá-la, com as palavras que um usuário usaria no pedido.
- **Corpo:** indica quais ferramentas chamar e em que ordem, usando pelo menos duas do servidor do grupo, o que fazer quando uma falhar ou não encontrar nada e o formato exato da resposta. A skill pode combinar também ferramentas dos servidores do mural. As ferramentas são citadas pelo nome que o agente vê, com a chave do servidor como prefixo, como `receitas_search_recipes`.

Depois de testada, a skill deve ser publicada na aba Skills do mural. As skills dos outros grupos ficam no mesmo lugar e podem ser copiadas para a pasta `skills/`, cada uma na própria pasta, desde que os servidores que elas usam também estejam em `servers`.

A ferramenta `load_skill` lê o `SKILL.md` pelo nome da skill, e o catálogo, que vai na mensagem de sistema, inclui o nome e a descrição de todas as skills da pasta `skills/`, as do grupo e as copiadas do mural.

In [ ]:
@tool
def load_skill(name: str) -> str:
    """Carrega as instruções completas de uma skill pelo nome."""
    return Path("skills", name, "SKILL.md").read_text()


metadata = [yaml.safe_load(path.read_text().split("---")[1]) for path in sorted(Path("skills").glob("*/SKILL.md"))]
catalog = "\n".join(f"- {meta['name']}: {meta['description']}" for meta in metadata)
print(catalog)

### Catálogo

Tabela com o servidor, o nome, a descrição e os argumentos de cada ferramenta disponível, seguida de uma avaliação das descrições dos servidores do mural: quais deixam claro quando a ferramenta deve ser usada e quais são ambíguas ou se sobrepõem às do servidor do grupo.

### Agente

Agente criado com `create_agent`, com as ferramentas de todos os conectores mais `load_skill` e com o catálogo de skills na mensagem de sistema. Devem ser feitas pelo menos sete perguntas, mostrando para cada uma as ferramentas chamadas e a resposta:

- pelo menos três que dependam dos servidores do mural;
- uma que precise combinar ferramentas do servidor do grupo com as de um servidor do mural;
- uma que deva carregar a skill do grupo e uma que deva carregar uma skill do mural;
- uma que nenhum servidor responda.

In [ ]:
async with adapter:
    tools = await adapter.list_tools()

# system_prompt = ""

# agent = create_agent(model, tools=tools + [load_skill], system_prompt=system_prompt)